# Tarea 3 - mapeo

### OBJETIVO: Dibujar todas las cicloestaciones disponibles en el catálogo de cicloestaciones del dataset de ecobicis y colocar en el marcador la siguiente información:

- Nombre de la ciclo estación
- Total histórico de viajes ( utilizar solo el dataset visto en clase )

### LIBRERIAS

In [3]:
import pandas as pd
import numpy as np
import os
import gc ##Limpia mi memoria RAM
import plotly.express as px
import plotly.io as pio
import folium
pio.renderers.default = 'notebook'

#### PARTE 1: RECUPERACION DE LOS DATOS

In [6]:
!ls -l

total 24
drwxr-xr-x 2 alex alex 4096 Sep  5 21:43 2020-2022
drwxr-xr-x 2 alex alex 4096 Sep  5 21:15 datos
drwxr-xr-x 7 alex alex 4096 Sep  5 01:15 diplomado_env
drwxr-xr-x 3 alex alex 4096 Sep  6 21:02 notebooks
drwxr-xr-x 3 alex alex 4096 Sep  6 20:56 proyecto_final
drwxr-xr-x 2 alex alex 4096 Sep  5 00:47 tareas


In [5]:
%cd ../

/home/alex/DiplomadoCD/Modulo1


In [7]:
# LEER EL ARCHIVO DE DATOS
# !tar -xvf ecobici.tar.gz -C datos

tar: ecobici.tar.gz: Cannot open: No such file or directory
tar: Error is not recoverable: exiting now


#### Tengo dos carpetas, una con los datos de 2017-2019 y la otra de 2020-2022

In [8]:
# variable del catalogo de estaciones
cat_estaciones = pd.read_csv("datos/estaciones-de-ecobici.csv")

### Verificacion datos, catalogo de estaciones

In [30]:
print(cat_estaciones.columns.tolist())

['ID', 'Nombre', 'Dirección', 'Número', 'Código postal', 'districtCode', 'Colonia', 'altitude', 'nearbyStations/0', 'location/lat', 'location/lon', 'Tipo de estación', 'nearbyStations/1', 'nearbyStations/2', 'nearbyStations/3', 'nearbyStations/4', 'nearbyStations/5', 'punto_geo']


In [31]:
cat_estaciones.head(5)

,ID,Nombre,Dirección,Número,Código postal,districtCode,Colonia,altitude,nearbyStations/0,location/lat,location/lon,Tipo de estación,nearbyStations/1,nearbyStations/2,nearbyStations/3,nearbyStations/4,nearbyStations/5,punto_geo
0,416,416 RECREO-PARROQUÍA,416 - Recreo-Parroquía,S/N,3104.0,1,Ampliación Granada,NaN,390,19.371308,-99.177607,BIKE,391.0,415.0,NaN,NaN,NaN,"19.371308,-99.177607"
1,440,440 REAL DE MAYORAZGO-PUENTE XOCO,440 - Real De Mayorazgo-Puente Xoco,S/N,3330.0,1,Ampliación Granada,NaN,433,19.360650,-99.168669,BIKE,439.0,441.0,NaN,NaN,NaN,"19.36065,-99.168669"
2,70,70 PARQUE MEXICO-MICHOACAN,070 - Parque México - Michoacan,S/N,6100.0,1,Ampliación Granada,NaN,69,19.411272,-99.169718,"BIKE,TPV",71.0,76.0,NaN,NaN,NaN,"19.411272,-99.169718"
3,325,325 PROVIDENCIA -AVENIDA DEL VALLE,325 - Providencia -Avenida Del Valle,S/N,3100.0,1,Ampliación Granada,NaN,316,19.388178,-99.169689,BIKE,335.0,336.0,NaN,NaN,NaN,"19.3881779,-99.1696894"
4,252,252 VICTOR HUGO-SHAKESPEARE,252 - Víctor Hugo-Shakespeare,S/N,11510.0,2,Anzures,NaN,249,19.427575,-99.177020,BIKE,251.0,253.0,NaN,NaN,NaN,"19.427575,-99.17702"


In [32]:
cat_estaciones = cat_estaciones.rename(columns={
    "ID": "id_estacion",
    "Nombre": "nombre",
    "location/lat": "latitud",
    "location/lon": "longitud"
})

In [33]:
cat_estaciones["latitud"] = pd.to_numeric(cat_estaciones["latitud"], errors="coerce")
cat_estaciones["longitud"] = pd.to_numeric(cat_estaciones["longitud"], errors="coerce")

In [12]:
#variable datos del 2017-2019
path = "datos"

lst_files = os.listdir("datos")

lst_files = [ f for f in lst_files if f[:2] == '20' ]

lst_df = []
for id_ in range(len(lst_files )):
    print("Leyendo archivo:", id_ )
    lst_df.append( pd.read_csv(os.path.join( path ,lst_files[id_] )) )

Leyendo archivo: 0


/tmp/ipykernel_24053/2415323979.py:11: DtypeWarning: Columns (0: Bici) have mixed types. Specify dtype option on import or set low_memory=False.
  lst_df.append( pd.read_csv(os.path.join( path ,lst_files[id_] )) )


Leyendo archivo: 1
Leyendo archivo: 2
Leyendo archivo: 3
Leyendo archivo: 4
Leyendo archivo: 5
Leyendo archivo: 6
Leyendo archivo: 7
Leyendo archivo: 8
Leyendo archivo: 9
Leyendo archivo: 10
Leyendo archivo: 11
Leyendo archivo: 12
Leyendo archivo: 13
Leyendo archivo: 14


/tmp/ipykernel_24053/2415323979.py:11: DtypeWarning: Columns (0: Bici, 1: Ciclo_Estacion_Arribo, 2: Unnamed: 9) have mixed types. Specify dtype option on import or set low_memory=False.
  lst_df.append( pd.read_csv(os.path.join( path ,lst_files[id_] )) )


Leyendo archivo: 15
Leyendo archivo: 16
Leyendo archivo: 17
Leyendo archivo: 18
Leyendo archivo: 19
Leyendo archivo: 20
Leyendo archivo: 21
Leyendo archivo: 22
Leyendo archivo: 23


In [13]:
df_viajes = pd.concat( lst_df , ignore_index=True)

In [14]:
df_viajes.info()

<class 'pandas.DataFrame'>
RangeIndex: 17104573 entries, 0 to 17104572
Data columns (total 10 columns):
 #   Column                 Dtype 
---  ------                 ----- 
 0   Genero_Usuario         str   
 1   Edad_Usuario           int64 
 2   Bici                   object
 3   Ciclo_Estacion_Retiro  int64 
 4   Fecha_Retiro           str   
 5   Hora_Retiro            str   
 6   Ciclo_Estacion_Arribo  object
 7   Fecha_Arribo           str   
 8   Hora_Arribo            str   
 9   Unnamed: 9             str   
dtypes: int64(2), object(2), str(6)
memory usage: 1.3+ GB


In [15]:
df_viajes.shape

(17104573, 10)

In [16]:
#Como borrar memoria
del lst_df
import gc
gc.collect()

475

#### EXPLORACION DE LOS DATOS

In [18]:
df_viajes.dtypes

Genero_Usuario              str
Edad_Usuario              int64
Bici                     object
Ciclo_Estacion_Retiro     int64
Fecha_Retiro                str
Hora_Retiro                 str
Ciclo_Estacion_Arribo    object
Fecha_Arribo                str
Hora_Arribo                 str
Unnamed: 9                  str
dtype: object

#### Vamos a trabajar con una Bola estadistica debido a mi limitacion de hardware, sera con un 10% y una semilla 

In [19]:
df_muestra = df_viajes.sample(frac=0.10, random_state=42)

# Limpaimos RAM
del df_viajes
gc.collect()

df_viajes = df_muestra

print(f"Tamaño de la bola estadistica: {len(df_viajes)} filas")

Tamaño de la bola estadistica: 1710457 filas


In [21]:
# Ver el porcentaje de valores nulos por columna
(df_viajes.isnull().sum() / len(df_viajes)) * 100

Genero_Usuario             0.0
Edad_Usuario               0.0
Bici                       0.0
Ciclo_Estacion_Retiro      0.0
Fecha_Retiro               0.0
Hora_Retiro                0.0
Ciclo_Estacion_Arribo      0.0
Fecha_Arribo               0.0
Hora_Arribo                0.0
Unnamed: 9               100.0
dtype: float64

#### Limpieza de los datos

In [22]:
### Celda de limpieza
# Convertir a int32 la edad usuario
df_viajes['Edad_Usuario'] = df_viajes['Edad_Usuario'].astype('int32')
# Todos los ids de las bicis a string
df_viajes['Bici'] = df_viajes['Bici'].astype(str)
# Eliminamos el registro "raro"
df_viajes = df_viajes[df_viajes['Unnamed: 9'].isna()].reset_index(drop=True)
# Todos los ids de ciclo estaciones a int32
df_viajes['Ciclo_Estacion_Retiro'] = df_viajes['Ciclo_Estacion_Retiro'].astype('int32')
df_viajes['Ciclo_Estacion_Arribo'] = df_viajes['Ciclo_Estacion_Arribo'].astype('int32')
# Eliminamos columna sin nombre
df_viajes.drop(columns=['Unnamed: 9'],inplace=True)

In [23]:
# Eliminar registros con fechas vacias
print(df_viajes.shape)
df_viajes = df_viajes[~df_viajes['Fecha_Retiro'].isna()].reset_index(drop=True)
print(df_viajes.shape)
df_viajes = df_viajes[~df_viajes['Fecha_Arribo'].isna()].reset_index(drop=True)
print(df_viajes.shape)

(1710457, 9)
(1710457, 9)
(1710457, 9)


In [24]:
df_viajes.sample(6)

,Genero_Usuario,Edad_Usuario,Bici,Ciclo_Estacion_Retiro,Fecha_Retiro,Hora_Retiro,Ciclo_Estacion_Arribo,Fecha_Arribo,Hora_Arribo
1362112,F,32,2403,187,15/03/2019,17:47:11,40,15/03/2019,18:01:30
296834,M,30,8401,287,12/03/2019,19:53:58,66,12/03/2019,20:16:56
1381772,F,34,10530,337,14/07/2019,18:54:42,302,14/07/2019,19:02:52
101344,M,42,8246,1,15/06/2018,15:01:50,5,15/06/2018,15:05:04
1329678,M,29,10218,461,14/08/2018,18:30:55,228,14/08/2018,18:46:41
1098657,M,25,3945,151,05/02/2018,18:30:03,170,05/02/2018,18:44:23


In [25]:
df_viajes['Fecha_Arribo'] = pd.to_datetime(df_viajes['Fecha_Arribo'], format='%d/%m/%Y', errors='coerce')
df_viajes['Fecha_Retiro'] = pd.to_datetime(df_viajes['Fecha_Retiro'], format='%d/%m/%Y', errors='coerce')

In [26]:
df_viajes.dtypes

Genero_Usuario                      str
Edad_Usuario                      int32
Bici                                str
Ciclo_Estacion_Retiro             int32
Fecha_Retiro             datetime64[us]
Hora_Retiro                         str
Ciclo_Estacion_Arribo             int32
Fecha_Arribo             datetime64[us]
Hora_Arribo                         str
dtype: object

In [27]:
df_viajes['Fecha_Retiro'].min(), df_viajes['Fecha_Retiro'].max()

(Timestamp('2016-12-02 00:00:00'), Timestamp('2019-10-31 00:00:00'))

In [28]:
df_viajes['Fecha_Arribo'].min(), df_viajes['Fecha_Arribo'].max()

(Timestamp('2017-11-01 00:00:00'), Timestamp('2019-10-31 00:00:00'))

In [29]:
for c in df_viajes.columns:
    print(c, df_viajes[c].map(type).unique().tolist() )

Genero_Usuario [<class 'str'>]
Edad_Usuario [<class 'int'>]
Bici [<class 'str'>]
Ciclo_Estacion_Retiro [<class 'int'>]
Fecha_Retiro [<class 'pandas.Timestamp'>]
Hora_Retiro [<class 'str'>]
Ciclo_Estacion_Arribo [<class 'int'>]
Fecha_Arribo [<class 'pandas.Timestamp'>]
Hora_Arribo [<class 'str'>]


### INTEGRACION

In [34]:
retiros = df_viajes["Ciclo_Estacion_Retiro"].value_counts().rename("retiros")
arribos = df_viajes["Ciclo_Estacion_Arribo"].value_counts().rename("arribos")
##CREAMOS EL HISTORICO DE VIAJES POR ESTACION
historico = pd.concat([retiros, arribos], axis=1).fillna(0)
historico["total_historico"] = historico["retiros"] + historico["arribos"]
historico = historico.reset_index().rename(columns={"index": "id_estacion"})

In [35]:
estaciones_map = cat_estaciones.merge(
    historico,
    on="id_estacion",
    how="left"
)
##UNIMOS EL CATALOGO DE ESTACIONES CON EL HISTORICO DE VIAJES
estaciones_map["total_historico"] = estaciones_map["total_historico"].fillna(0)

In [ ]:
##INTEGRAMOS

m = folium.Map(location=[19.4326, -99.1332], zoom_start=11)

for _, row in estaciones_map.iterrows():
    folium.Marker(
        location=[row["latitud"], row["longitud"]],
        popup=f"""
        <b>{row["nombre"]}</b><br>
        Total de viajes: {int(row["total_historico"])}
        """,
        tooltip=row["nombre"]
    ).add_to(m)


m

NOTA: COMO NOTA DEBE DE DECIRSE QUE SE USO UNA BOLA ESTADISTICA (SE COLOCO UNA SEMILLA PARA REPRODUCIR LOS DATOS) DEBIDO A LAS LIMITACIONES COMPUTACIONALES DE MI EQUIPO, YA QUE AL TOMAR EL DATASET COMPLETO DE 2017-2019 MI MAQUINA NO JECUTA EL CODIGO.  